In [1]:
import os
from pathlib import Path

from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID";
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

**Entrenamiento de autoencoder SPECT**

In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


from pathlib import Path
import subprocess

ROOT = find_repo_root()
os.chdir(ROOT)
log_path = ROOT / "exp_nm_s1_spect.log"

ROOT = find_repo_root()
os.chdir(ROOT)
log_path = ROOT / "exp_nm_s1_spect.log"
with open(log_path, "w") as log:
    proc = subprocess.Popen(
        ["bash", "scripts/run_experiment.sh", "s1_spect", "exp_nm_s1_spect"],
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        start_new_session=True,  # no muere al cerrar la celda
    )
print(f"Lanzado PID={proc.pid}")
print(f"Log: {log_path}")


**Extraccion de latentes a partir de encoder entrenado**

In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os
import sys
import subprocess
from pathlib import Path

ROOT = find_repo_root()
env = os.environ.copy()
env["ILCLDM_EXP"] = "exp_nm_s1_spect/"

with open(ROOT / "encode_latents.log", "w") as log:
    proc = subprocess.Popen(
        [
            sys.executable,  # ← mismo Python que el kernel
            "-c",
            "from utils import seed_torch; from main import encoding; "
            "seed_torch(); encoding()",
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        env=env,
        start_new_session=True,
    )

print(f"Lanzado PID={proc.pid}")
print(f"Python: {sys.executable}")
print(f"Log: {ROOT / 'encode_latents.log'}")


**Replicar entrenamiento pero esta vez para MRI**

In [2]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


from pathlib import Path
import subprocess

ROOT = find_repo_root()
os.chdir(ROOT)
log_path = ROOT / "exp_nm_s1_spect.log"

ROOT = find_repo_root()
os.chdir(ROOT)
log_path = ROOT / "exp_nm_s1_mri.log"
with open(log_path, "w") as log:
    proc = subprocess.Popen(
        ["bash", "scripts/run_experiment.sh", "s1_mri", "exp_nm_s1_mri"],
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        start_new_session=True,  # no muere al cerrar la celda
    )
print(f"Lanzado PID={proc.pid}")
print(f"Log: {log_path}")


Lanzado PID=276800
Log: /workspace/IL-CLDM/exp_nm_s1_mri.log


**Running the E3 (baseline) que es el tradicional sin preentrenar encoder y embeddings MRI**

In [6]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os
import subprocess
from pathlib import Path

ROOT = find_repo_root()

# AAE congelado de E1 (config.ILCLDM_AAE_CHECKPOINT; sin copiar a exp_nm_s2_baseline)
aae_spect = ROOT / "result/exp_nm_s1_spect/AAE.pth.tar"
if not aae_spect.is_file():
    raise FileNotFoundError(f"Falta el AAE de E1: {aae_spect}")

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "7"
env["ILCLDM_AAE_CHECKPOINT"] = str(aae_spect.resolve())

# Unet y CSVs se guardan en result/exp_nm_s2_baseline/ (ILCLDM_EXP del script s2_baseline)

# Opcional: acelerar E3
# env["ILCLDM_VALIDATION_INTERVAL"] = "10"
# env["ILCLDM_SAMPLING_STEPS"] = "25"

with open(ROOT / "exp_nm_s2_baseline.log", "w") as log:
    proc = subprocess.Popen(
        ["bash", "scripts/run_experiment.sh", "s2_baseline", "exp_nm_s2_baseline"],
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        env=env,
        start_new_session=True,
    )

print("E3 PID", proc.pid)
print("GPU", env["CUDA_VISIBLE_DEVICES"])
print("AAE (read-only)", env["ILCLDM_AAE_CHECKPOINT"])
print("UNet output dir", ROOT / "result/exp_nm_s2_baseline")
print("Log", ROOT / "exp_nm_s2_baseline.log")


E3 PID 282894
GPU 7
AAE (read-only) /workspace/IL-CLDM/result/exp_nm_s1_spect/AAE.pth.tar
UNet output dir /workspace/IL-CLDM/result/exp_nm_s2_baseline
Log /workspace/IL-CLDM/exp_nm_s2_baseline.log


**Ejecucion de experimento con autoencoder MRI entrenable**

In [2]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os, subprocess
from pathlib import Path

ROOT = find_repo_root()
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "7"  # otra GPU si E5 corre en paralelo
env["ILCLDM_AAE_CHECKPOINT"] = str((ROOT / "result/exp_nm_s1_spect/AAE.pth.tar").resolve())
env["ILCLDM_MRI_ENCODER_CHECKPOINT"] = str(
    (ROOT / "result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar").resolve()
)
env["ILCLDM_MRI_ENCODER_FINETUNE_LR_SCALE"] = "0.25"

with open(ROOT / "exp_nm_s2_mri_finetune.log", "w") as log:
    proc = subprocess.Popen(
        ["bash", "scripts/run_experiment.sh", "s2_mri_finetune", "exp_nm_s2_mri_finetune"],
        stdout=log, stderr=subprocess.STDOUT,
        cwd=str(ROOT), env=env, start_new_session=True,
    )
print("E4 PID", proc.pid)


E4 PID 6760


**Ejecucion de experimento con autoencoder MRI congelado**

**Inferencia stage-2 (E3/E5/E4): reconstrucciones, latentes, ML train→test**

Ver `scripts/run_stage2_inference.sh` y `RUNBOOK.md` sección 7.

In [1]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os, subprocess
from pathlib import Path

ROOT = find_repo_root()
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "6"  # GPU libre
env["ILCLDM_AAE_CHECKPOINT"] = str((ROOT / "result/exp_nm_s1_spect/AAE.pth.tar").resolve())
env["ILCLDM_MRI_ENCODER_CHECKPOINT"] = str(
    (ROOT / "result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar").resolve()
)

with open(ROOT / "exp_nm_s2_mri_frozen.log", "w") as log:
    proc = subprocess.Popen(
        ["bash", "scripts/run_experiment.sh", "s2_mri_frozen", "exp_nm_s2_mri_frozen"],
        stdout=log, stderr=subprocess.STDOUT,
        cwd=str(ROOT), env=env, start_new_session=True,
    )
print("E5 PID", proc.pid)


E5 PID 6473


In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os
import sys
import subprocess
from pathlib import Path

ROOT = find_repo_root()
os.chdir(ROOT)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "6"
env["ILCLDM_AAE_CHECKPOINT"] = str((ROOT / "result/exp_nm_s1_spect/AAE.pth.tar").resolve())
env["RUN_ID"] = "nm_stage2_v1"

# Recuperar encoder E4 si no existe (corrida previa sin guardado)
finetuned = ROOT / "result/exp_nm_s2_mri_finetune/MRI_Encoder_finetuned.pth.tar"
if not finetuned.is_file():
    env["ILCLDM_MRI_ENCODER_CHECKPOINT"] = str(
        (ROOT / "result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar").resolve()
    )
    with open(ROOT / "recover_finetune_encoder.log", "w") as log:
        subprocess.Popen(
            [sys.executable, "scripts/recover_finetune_mri_encoder.py", "--epochs", "50"],
            stdout=log, stderr=subprocess.STDOUT, cwd=str(ROOT), env=env, start_new_session=True,
        )
    print("Encoder recovery launched; see recover_finetune_encoder.log")

with open(ROOT / "stage2_inference.log", "w") as log:
    proc = subprocess.Popen(
        ["bash", "scripts/run_stage2_gpu6_pipeline.sh"],
        stdout=log, stderr=subprocess.STDOUT, cwd=str(ROOT), env=env, start_new_session=True,
    )
print(f"Inference batch PID={proc.pid}, RUN_ID={env['RUN_ID']}")
print(f"Log: {ROOT / 'stage2_inference.log'}")


**Entrenamiento `h05_roi3ch`:** warm-start desde `abl_hight05` + UNet con 3.er canal ROI (high-t 50% como hight05)

In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os
import subprocess
from datetime import datetime
from pathlib import Path

ROOT = find_repo_root()
os.chdir(ROOT)

# Prerrequisitos E1 + E2 + máscara ROI (v2)
aae_spect = ROOT / "result/exp_nm_s1_spect/AAE.pth.tar"
mri_encoder = ROOT / "result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar"
roi_mask = ROOT / "data/mask_basal_ganglia.nii.gz"
latent_dir = ROOT / "data/latent_Abeta"

for path, label in [
    (aae_spect, "AAE SPECT (E1)"),
    (mri_encoder, "MRI encoder (E2)"),
    (roi_mask, "ROI basal ganglia"),
]:
    if not path.is_file():
        raise FileNotFoundError(f"Falta {label}: {path}")

if not latent_dir.is_dir() or not any(latent_dir.glob("*.nii.gz")):
    raise FileNotFoundError(
        f"Faltan latentes en {latent_dir}. Ejecuta antes E1 (s1_spect + encoding)."
    )

log_path = ROOT / "exp_nm_s2_mri_frozen_v2.log"
pid_path = ROOT / "exp_nm_s2_mri_frozen_v2.pid"
out_dir = ROOT / "result/exp_nm_s2_mri_frozen_v2"

env = os.environ.copy()
env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
env["CUDA_VISIBLE_DEVICES"] = "3"
env["ILCLDM_AAE_CHECKPOINT"] = str(aae_spect.resolve())
env["ILCLDM_MRI_ENCODER_CHECKPOINT"] = str(mri_encoder.resolve())

with open(log_path, "a") as log:
    log.write(f"\n=== E5 v2 frozen launch {datetime.now().isoformat(timespec='seconds')} ===\n")

# nohup: el entrenamiento sigue tras cerrar SSH o el kernel del notebook
with open(log_path, "a") as log:
    proc = subprocess.Popen(
        [
            "nohup",
            "bash",
            "scripts/run_experiment.sh",
            "s2_mri_frozen_v2",
            "exp_nm_s2_mri_frozen_v2",
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        env=env,
        start_new_session=True,
    )

pid_path.write_text(str(proc.pid))

print(f"E5 v2 (frozen) lanzado con nohup — PID={proc.pid}")
print("GPU: ", env["CUDA_VISIBLE_DEVICES"])
print("Modo: s2_mri_frozen_v2 (latent loss, high-t sampling, ROI aux, early stop latente)")
print("AAE:", env["ILCLDM_AAE_CHECKPOINT"])
print("MRI encoder:", env["ILCLDM_MRI_ENCODER_CHECKPOINT"])
print("Salidas:", out_dir)
print("Log:", log_path)
print("PID file:", pid_path)
print("\nMonitorizar:")
print(f"  tail -f {log_path}")
print(f"  ps -p {proc.pid} -o pid,cmd,etime")


E5 v2 (frozen) lanzado con nohup — PID=121345
GPU:  3
Modo: s2_mri_frozen_v2 (latent loss, high-t sampling, ROI aux, early stop latente)
AAE: /workspace/IL-CLDM/result/exp_nm_s1_spect/AAE.pth.tar
MRI encoder: /workspace/IL-CLDM/result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar
Salidas: /workspace/IL-CLDM/result/exp_nm_s2_mri_frozen_v2
Log: /workspace/IL-CLDM/exp_nm_s2_mri_frozen_v2.log
PID file: /workspace/IL-CLDM/exp_nm_s2_mri_frozen_v2.pid

Monitorizar:
  tail -f /workspace/IL-CLDM/exp_nm_s2_mri_frozen_v2.log
  ps -p 121345 -o pid,cmd,etime


# From here I will test the performance of each improvement independently
* Ablation: latent loss with 0.2 of weight

In [4]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


# here is for run the inference with the ablation of the latent
# loss

import os
import sys
import subprocess
from pathlib import Path

ROOT = find_repo_root()
os.chdir(ROOT)

GPU = "6"  # cambia al índice que quieras
RUN_ID = "h05_aux5"
EXP = "exp_nm_s2_frozen_h05_aux5"

env = os.environ.copy()
env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
env["CUDA_VISIBLE_DEVICES"] = GPU
env["ILCLDM_AAE_CHECKPOINT"] = str((ROOT / "result/exp_nm_s1_spect/AAE.pth.tar").resolve())

log_path = ROOT / "logs" / f"infer_{RUN_ID}_gpu{GPU}.log"
pid_path = ROOT / "logs" / f"infer_{RUN_ID}_gpu{GPU}.pid"
log_path.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    "nohup",
    sys.executable, "inference.py",
    "--exp", EXP,
    "--mri-encoder-mode", "frozen",
    "--mri-encoder-checkpoint",
    str((ROOT / "result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar").resolve()),
    "--run-id", RUN_ID,
    "--save-synthetic-spect",
    "--splits", "train", "test",
]

with open(log_path, "w") as log:
    proc = subprocess.Popen(
        cmd,
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        env=env,
        start_new_session=True,
    )

print(f"PID={proc.pid}")
print(f"GPU={GPU}")
print(f"Log: {log_path}")
print(f"Salida: {ROOT / 'result' / EXP / 'inference' / RUN_ID}")


PID=458299
GPU=6
Log: /workspace/IL-CLDM/logs/infer_h05_aux5_gpu6.log
Salida: /workspace/IL-CLDM/result/exp_nm_s2_frozen_h05_aux5/inference/h05_aux5


# Experiments taking now as baseline the hight05 model

## In this new section, new experiments are considered:
1. Training without label
2. Training with ROI as 3 channel in Unet
3. Include hight05 and auxiliar image loss
4. Augmentation of the exposure over > 700 t steps with 80% of probabilities

In [1]:
# h05_roi3ch: warm-start abl_hight05 + UNet con 3.er canal ROI

from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path

ROOT = find_repo_root()
os.chdir(ROOT)

REGISTRY_KEY = "h05_roi3ch"
GPU = "7"  # cambia al índice que quieras
EXP = "exp_nm_s2_frozen_h05_roi3ch"

aae_spect = ROOT / "result/exp_nm_s1_spect/AAE.pth.tar"
mri_encoder = ROOT / "result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar"
roi_mask = ROOT / "data/mask_basal_ganglia.nii.gz"
latent_dir = ROOT / "data/latent_Abeta"
hight05_unet = ROOT / "result/exp_nm_s2_frozen_abl_hight05/Unet.pth.tar"

for path, label in [
    (aae_spect, "AAE SPECT (E1)"),
    (mri_encoder, "MRI encoder (E2)"),
    (roi_mask, "ROI basal ganglia (3er canal UNet)"),
    (hight05_unet, "UNet abl_hight05 (warm-start)"),
]:
    if not path.is_file():
        raise FileNotFoundError(f"Falta {label}: {path}")

if not latent_dir.is_dir() or not any(latent_dir.glob("*.nii.gz")):
    raise FileNotFoundError(
        f"Faltan latentes en {latent_dir}. Ejecuta antes E1 (s1_spect + encoding)."
    )

log_path = ROOT / "logs" / f"train_{REGISTRY_KEY}_gpu{GPU}.log"
pid_path = ROOT / "logs" / f"train_{REGISTRY_KEY}_gpu{GPU}.pid"
log_path.parent.mkdir(parents=True, exist_ok=True)
out_dir = ROOT / "result" / EXP

env = os.environ.copy()
env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
env["ILCLDM_AAE_CHECKPOINT"] = str(aae_spect.resolve())

with open(log_path, "a") as log:
    log.write(f"\n=== {REGISTRY_KEY} launch {datetime.now().isoformat(timespec='seconds')} ===\n")

with open(log_path, "a") as log:
    proc = subprocess.Popen(
        [
            "nohup",
            sys.executable,
            "scripts/run_ablation.py",
            REGISTRY_KEY,
            "--gpu",
            GPU,
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        env=env,
        start_new_session=True,
    )

pid_path.write_text(str(proc.pid))

print(f"{REGISTRY_KEY} lanzado con nohup — PID={proc.pid}")
print(f"GPU={GPU}")
print("Modo: base hight05 + ILCLDM_UNET_ROI_CHANNEL=1, warm-start abl_hight05")
print("Warm-start:", hight05_unet)
print("Salidas:", out_dir)
print("Log:", log_path)
print("PID file:", pid_path)
print("\nMonitorizar:")
print(f"  tail -f {log_path}")
print(f"  ps -p {proc.pid} -o pid,cmd,etime")

h05_roi3ch lanzado con nohup — PID=440659
GPU=7
Modo: base hight05 + ILCLDM_UNET_ROI_CHANNEL=1, warm-start abl_hight05
Warm-start: /workspace/IL-CLDM/result/exp_nm_s2_frozen_abl_hight05/Unet.pth.tar
Salidas: /workspace/IL-CLDM/result/exp_nm_s2_frozen_h05_roi3ch
Log: /workspace/IL-CLDM/logs/train_h05_roi3ch_gpu7.log
PID file: /workspace/IL-CLDM/logs/train_h05_roi3ch_gpu7.pid

Monitorizar:
  tail -f /workspace/IL-CLDM/logs/train_h05_roi3ch_gpu7.log
  ps -p 440659 -o pid,cmd,etime


Para experimento no label

In [2]:
# h05_nolabel: warm-start abl_hight05, sin label en entrenamiento (inferencia ya no usa label)

from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path

ROOT = find_repo_root()
os.chdir(ROOT)

REGISTRY_KEY = "h05_nolabel"
GPU = "5"  # cambia al índice que quieras
EXP = "exp_nm_s2_frozen_h05_nolabel"

aae_spect = ROOT / "result/exp_nm_s1_spect/AAE.pth.tar"
mri_encoder = ROOT / "result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar"
latent_dir = ROOT / "data/latent_Abeta"
hight05_unet = ROOT / "result/exp_nm_s2_frozen_abl_hight05/Unet.pth.tar"

for path, label in [
    (aae_spect, "AAE SPECT (E1)"),
    (mri_encoder, "MRI encoder (E2)"),
    (hight05_unet, "UNet abl_hight05 (warm-start)"),
]:
    if not path.is_file():
        raise FileNotFoundError(f"Falta {label}: {path}")

if not latent_dir.is_dir() or not any(latent_dir.glob("*.nii.gz")):
    raise FileNotFoundError(
        f"Faltan latentes en {latent_dir}. Ejecuta antes E1 (s1_spect + encoding)."
    )

log_path = ROOT / "logs" / f"train_{REGISTRY_KEY}_gpu{GPU}.log"
pid_path = ROOT / "logs" / f"train_{REGISTRY_KEY}_gpu{GPU}.pid"
log_path.parent.mkdir(parents=True, exist_ok=True)
out_dir = ROOT / "result" / EXP

env = os.environ.copy()
env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
env["ILCLDM_AAE_CHECKPOINT"] = str(aae_spect.resolve())

with open(log_path, "a") as log:
    log.write(f"\n=== {REGISTRY_KEY} launch {datetime.now().isoformat(timespec='seconds')} ===\n")

with open(log_path, "a") as log:
    proc = subprocess.Popen(
        [
            "nohup",
            sys.executable,
            "scripts/run_ablation.py",
            REGISTRY_KEY,
            "--gpu",
            GPU,
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        env=env,
        start_new_session=True,
    )

pid_path.write_text(str(proc.pid))

print(f"{REGISTRY_KEY} lanzado con nohup — PID={proc.pid}")
print(f"GPU={GPU}")
print("Modo: base hight05 + ILCLDM_USE_TRAIN_LABEL=0, warm-start abl_hight05")
print("Warm-start:", hight05_unet)
print("Salidas:", out_dir)
print("Log:", log_path)
print("PID file:", pid_path)
print("\nMonitorizar:")
print(f"  tail -f {log_path}")
print(f"  ps -p {proc.pid} -o pid,cmd,etime")

h05_nolabel lanzado con nohup — PID=442113
GPU=5
Modo: base hight05 + ILCLDM_USE_TRAIN_LABEL=0, warm-start abl_hight05
Warm-start: /workspace/IL-CLDM/result/exp_nm_s2_frozen_abl_hight05/Unet.pth.tar
Salidas: /workspace/IL-CLDM/result/exp_nm_s2_frozen_h05_nolabel
Log: /workspace/IL-CLDM/logs/train_h05_nolabel_gpu5.log
PID file: /workspace/IL-CLDM/logs/train_h05_nolabel_gpu5.pid

Monitorizar:
  tail -f /workspace/IL-CLDM/logs/train_h05_nolabel_gpu5.log
  ps -p 442113 -o pid,cmd,etime


Experiment for oversampling t > 700 with 80% of probabilities

In [3]:
# h05_ht08: warm-start abl_hight05 + high-t sampling 80% (t >= 700)

from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path

ROOT = find_repo_root()
os.chdir(ROOT)

REGISTRY_KEY = "h05_ht08"
GPU = "6"  # cambia al índice que quieras
EXP = "exp_nm_s2_frozen_h05_ht08"

aae_spect = ROOT / "result/exp_nm_s1_spect/AAE.pth.tar"
mri_encoder = ROOT / "result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar"
latent_dir = ROOT / "data/latent_Abeta"
hight05_unet = ROOT / "result/exp_nm_s2_frozen_abl_hight05/Unet.pth.tar"

for path, label in [
    (aae_spect, "AAE SPECT (E1)"),
    (mri_encoder, "MRI encoder (E2)"),
    (hight05_unet, "UNet abl_hight05 (warm-start)"),
]:
    if not path.is_file():
        raise FileNotFoundError(f"Falta {label}: {path}")

if not latent_dir.is_dir() or not any(latent_dir.glob("*.nii.gz")):
    raise FileNotFoundError(
        f"Faltan latentes en {latent_dir}. Ejecuta antes E1 (s1_spect + encoding)."
    )

log_path = ROOT / "logs" / f"train_{REGISTRY_KEY}_gpu{GPU}.log"
pid_path = ROOT / "logs" / f"train_{REGISTRY_KEY}_gpu{GPU}.pid"
log_path.parent.mkdir(parents=True, exist_ok=True)
out_dir = ROOT / "result" / EXP

env = os.environ.copy()
env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
env["ILCLDM_AAE_CHECKPOINT"] = str(aae_spect.resolve())

with open(log_path, "a") as log:
    log.write(f"\n=== {REGISTRY_KEY} launch {datetime.now().isoformat(timespec='seconds')} ===\n")

with open(log_path, "a") as log:
    proc = subprocess.Popen(
        [
            "nohup",
            sys.executable,
            "scripts/run_ablation.py",
            REGISTRY_KEY,
            "--gpu",
            GPU,
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        env=env,
        start_new_session=True,
    )

pid_path.write_text(str(proc.pid))

print(f"{REGISTRY_KEY} lanzado con nohup — PID={proc.pid}")
print(f"GPU={GPU}")
print("Modo: base hight05 + ILCLDM_HIGH_T_SAMPLE_PROB=0.8, warm-start abl_hight05")
print("Warm-start:", hight05_unet)
print("Salidas:", out_dir)
print("Log:", log_path)
print("PID file:", pid_path)
print("\nMonitorizar:")
print(f"  tail -f {log_path}")
print(f"  ps -p {proc.pid} -o pid,cmd,etime")

h05_ht08 lanzado con nohup — PID=444774
GPU=6
Modo: base hight05 + ILCLDM_HIGH_T_SAMPLE_PROB=0.8, warm-start abl_hight05
Warm-start: /workspace/IL-CLDM/result/exp_nm_s2_frozen_abl_hight05/Unet.pth.tar
Salidas: /workspace/IL-CLDM/result/exp_nm_s2_frozen_h05_ht08
Log: /workspace/IL-CLDM/logs/train_h05_ht08_gpu6.log
PID file: /workspace/IL-CLDM/logs/train_h05_ht08_gpu6.pid

Monitorizar:
  tail -f /workspace/IL-CLDM/logs/train_h05_ht08_gpu6.log
  ps -p 444774 -o pid,cmd,etime


From here is for the experiment that includes High05 exposure to t > 700 and image loss

In [1]:
# h05_aux5: warm-start abl_hight05 + high-t 50% + aux image loss cada 5 batches

from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path

ROOT = find_repo_root()
os.chdir(ROOT)

REGISTRY_KEY = "h05_aux5"
GPU = "7"  # cambia al índice que quieras
EXP = "exp_nm_s2_frozen_h05_aux5"

aae_spect = ROOT / "result/exp_nm_s1_spect/AAE.pth.tar"
mri_encoder = ROOT / "result/exp_nm_s1_mri/phase1_mri/checkpoints/MRI_AAE.pth.tar"
latent_dir = ROOT / "data/latent_Abeta"
hight05_unet = ROOT / "result/exp_nm_s2_frozen_abl_hight05/Unet.pth.tar"

for path, label in [
    (aae_spect, "AAE SPECT (E1)"),
    (mri_encoder, "MRI encoder (E2)"),
    (hight05_unet, "UNet abl_hight05 (warm-start)"),
]:
    if not path.is_file():
        raise FileNotFoundError(f"Falta {label}: {path}")

if not latent_dir.is_dir() or not any(latent_dir.glob("*.nii.gz")):
    raise FileNotFoundError(
        f"Faltan latentes en {latent_dir}. Ejecuta antes E1 (s1_spect + encoding)."
    )

log_path = ROOT / "logs" / f"train_{REGISTRY_KEY}_gpu{GPU}.log"
pid_path = ROOT / "logs" / f"train_{REGISTRY_KEY}_gpu{GPU}.pid"
log_path.parent.mkdir(parents=True, exist_ok=True)
out_dir = ROOT / "result" / EXP

env = os.environ.copy()
env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
env["ILCLDM_AAE_CHECKPOINT"] = str(aae_spect.resolve())

with open(log_path, "a") as log:
    log.write(f"\n=== {REGISTRY_KEY} launch {datetime.now().isoformat(timespec='seconds')} ===\n")

with open(log_path, "a") as log:
    proc = subprocess.Popen(
        [
            "nohup",
            sys.executable,
            "scripts/run_ablation.py",
            REGISTRY_KEY,
            "--gpu",
            GPU,
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        env=env,
        start_new_session=True,
    )

pid_path.write_text(str(proc.pid))

print(f"{REGISTRY_KEY} lanzado con nohup — PID={proc.pid}")
print(f"GPU={GPU}")
print("Modo: base hight05 + AUX_IMAGE_EVERY_N=5, AUX_IMAGE_LAMBDA=0.25, warm-start abl_hight05")
print("Warm-start:", hight05_unet)
print("Salidas:", out_dir)
print("Log:", log_path)
print("PID file:", pid_path)
print("\nMonitorizar:")
print(f"  tail -f {log_path}")
print(f"  ps -p {proc.pid} -o pid,cmd,etime")

h05_aux5 lanzado con nohup — PID=448834
GPU=7
Modo: base hight05 + AUX_IMAGE_EVERY_N=5, AUX_IMAGE_LAMBDA=0.25, warm-start abl_hight05
Warm-start: /workspace/IL-CLDM/result/exp_nm_s2_frozen_abl_hight05/Unet.pth.tar
Salidas: /workspace/IL-CLDM/result/exp_nm_s2_frozen_h05_aux5
Log: /workspace/IL-CLDM/logs/train_h05_aux5_gpu7.log
PID file: /workspace/IL-CLDM/logs/train_h05_aux5_gpu7.pid

Monitorizar:
  tail -f /workspace/IL-CLDM/logs/train_h05_aux5_gpu7.log
  ps -p 448834 -o pid,cmd,etime


In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


os.chdir(find_repo_root())
os.system("nohup python inference.py --run-conditioning-audit --exp exp_1 --run-id conditioning_full --splits train test --batch-size 1")


In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


os.chdir(find_repo_root())
os.system("nohup python code/run_phase1_mri_experiment.py --exp exp_2 --skip-train > mri_stage_1_embeddings.log 2>&1 &")


In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import pandas as pd

data = str(find_repo_root() / "result/exp_1/embeddings/conditioning_mri/conditioning_full/classification/classification_metrics.csv")
df = pd.read_csv(data)
df.head()


**Visualizing classification results**

In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    """Resolve IL-CLDM repo root from cwd (works in devcontainer or local clone)."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "config.py").is_file() and (candidate / "data_info" / "train.txt").is_file():
            return candidate
    raise RuntimeError(
        "Could not find IL-CLDM repo root. Run from the repo or a subdirectory."
    )


import json
import pandas as pd
# metrics_summary.json is a list of dicts, not a dict. Just load into DataFrame directly.
results_path = str(find_repo_root() / "result/exp_nm_s1_spect/embedding_analysis/classification/metrics_summary.json")

# Load the JSON file containing classification results (list of dicts)
with open(results_path, "r") as f:
    metrics = json.load(f)

# Convert the list of dicts to a DataFrame for easier visualization
results_df = pd.DataFrame(metrics)

# Optionally, set the model name or classifier name as the index (up to you)
# results_df = results_df.set_index("model")

# Display the summary table
results_df


,model,accuracy,balanced_accuracy,f1,roc_auc,classifier,n_neighbors,kernel,C
0,LogisticRegression,0.933333,0.944444,0.941176,0.995370,LogisticRegression,NaN,NaN,1.0
1,SVM_linear,0.933333,0.944444,0.941176,0.981481,SVC,NaN,linear,1.0
2,KNN_auto,0.800000,0.819444,0.812500,0.847222,KNeighborsClassifier,5.0,NaN,NaN
3,KNN_K2,0.700000,0.708333,0.727273,0.807870,KNeighborsClassifier,2.0,NaN,NaN
